<a href="https://colab.research.google.com/github/maierav/ai_oscp_neuro/blob/main/notebooks/crossscale_timescales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Comparing oddball responses across spiking and imaging — the timescale problem

Neuropixels, mesoscope (jGCaMP8s), and SLAP2 (iGluSnFR) all record the **same**
standard-oddball gratings, but through measurement processes with very different
temporal kinetics: a spike burst is over in ~150 ms, a jGCaMP8s transient decays
over ~1–2 s. A fixed post-stimulus window tuned to one modality captures a
different *fraction* of the response in another — so the same underlying effect
reads as different magnitudes across scales. This is a measurement artifact, not
biology.

**This notebook solves it empirically** — no deconvolution, no kernel-fitting.
It streams one-to-three sessions per modality, plots the trial- and
population-averaged response to the same gratings on a common time axis, and
derives per-modality integration windows directly from where each response
actually lives (the cumulative-response curve).

**Key methodological point:** population-*mean* imaging traces are dominated by
ongoing activity and give spurious peak times. The fair comparison restricts to
**responsive cells** (units/ROIs with a reliable evoked response). Once gated,
the timescales order sensibly (spikes fastest, jGCaMP8s slowest) and the
90%-cumulative-response windows nearly coincide (~0–1700 ms) — so a single
common window treats all three modalities fairly.


In [ ]:
#@title Install
!pip -q install pynwb h5py remfile requests pandas numpy matplotlib scipy

In [ ]:
#@title Streaming helpers
import numpy as np, pandas as pd, h5py, remfile, requests, re
from scipy import stats as ss
from scipy.ndimage import gaussian_filter1d

def s3_url(ds, aid, version="draft"):
    u=f"https://api.dandiarchive.org/api/dandisets/{ds}/versions/{version}/assets/{aid}/download/"
    return requests.get(u,allow_redirects=False,timeout=60).headers["Location"]
def col(g,c):
    v=g[c][:]; return np.array([x.decode() if isinstance(x,bytes) else x for x in v])

PRE,POST=0.3,2.0
GRID=np.arange(-PRE,POST,0.02)
BW=0.02; EDGES=np.arange(-PRE,POST+BW,BW); CEN=EDGES[:-1]+BW/2
print("ready")

In [ ]:
#@title Sessions (one–three per modality; all recorded the same gratings)
# Neuropixels (DANDI 001637), Mesoscope (001768), SLAP2 (001424)
SESSIONS = {
  "ecephys":  [("830851","9b9e8abe-7b43-47f1-b8e1-4114f87898a1"),
               ("830848","228c2c2e-1daf-4bf6-9f66-eb6b2bce5ba5"),
               ("830846","680d1c0c-e338-4d0b-ba29-4329436d2ae2")],
  "mesoscope":[("845342","6288e7b0-5b44-4660-b36d-c14d19220e2c"),
               ("837568","b425c043-ebf5-456c-83a9-1d99465224c6")],
  "slap2":    [("796630","b8c05ec0-0a74-46f5-956d-e82af3cc5d27"),
               ("803496","d23a03af-c3bd-4cf0-9492-6dca96fb201d"),
               ("801381","2ecafd40-29a6-422f-92b0-32f7e940c783")],
}
# Each imaging session runs several minutes (per-ROI responsiveness test). For a
# quick pass set ONE_PER_MODALITY=True to use just the first session of each.
ONE_PER_MODALITY = True
if ONE_PER_MODALITY:
    SESSIONS = {k:v[:1] for k,v in SESSIONS.items()}
print({k:[s for s,_ in v] for k,v in SESSIONS.items()})

In [ ]:
#@title Responsive-cell trace extractors
def ece_resp_trace(aid):
    """Population-mean PSTH over RESPONSIVE visual units (0°standard & 90°oddball)."""
    fh=h5py.File(remfile.File(s3_url("001637",aid)),"r")
    try:
        gO=fh["intervals"]["Standard mismatch block_presentations"];TT=col(gO,"TrialType");ts=gO["start_time"][:]
        t_std=ts[TT=="standard"];t_o90=ts[TT=="orientation_90"]
        U=fh["units"];st=U["spike_times"][:];sti=U["spike_times_index"][:]
        n=len(sti);qc=U["default_qc"][:] if "default_qc" in U else np.ones(n,bool)
        Eg=fh["general/extracellular_ephys/electrodes"];eloc=col(Eg,"location");egroup=col(Eg,"group_name")
        dev=col(U,"device_name");eci=U["extremum_channel_index"][:]
        groups=sorted(set(egroup),key=lambda gn:np.where(egroup==gn)[0][0])
        offs={gn:int(np.where(egroup==gn)[0][0]) for gn in groups};bl={gn:int((egroup==gn).sum()) for gn in groups}
        def d2g(d):
            for gn in groups:
                if d[-1].lower() in gn.lower(): return gn
            return groups[0]
        dmap={d:d2g(d) for d in set(dev)}
        # extremum_channel_index is PER-PROBE; offset into the stacked electrodes table
        u_loc=np.array([eloc[offs[dmap[dev[i]]]+min(int(eci[i]),bl[dmap[dev[i]]]-1)] for i in range(n)])
        vis=np.where(qc & np.array([bool(re.match("VIS",r)) for r in u_loc]))[0]
        def sp_(i): return st[(0 if i==0 else sti[i-1]):sti[i]]
        def rate(sp,times,w): lo=np.searchsorted(sp,times+w[0]);hi=np.searchsorted(sp,times+w[1]);return (hi-lo)/(w[1]-w[0])
        std_acc=np.zeros(len(CEN));o90_acc=np.zeros(len(CEN));nresp=0
        for u in vis:
            sp=sp_(u); ev=rate(sp,t_std,(0.045,0.295))-rate(sp,t_std,(-0.1,-0.005))
            if not (np.any(ev!=0) and ss.wilcoxon(ev).pvalue<0.05): continue
            nresp+=1
            for times,acc in [(t_std,std_acc),(t_o90,o90_acc)]:
                lo=np.searchsorted(sp,times.min()-PRE);hi=np.searchsorted(sp,times.max()+POST);sp2=sp[lo:hi]
                rel=(sp2[None,:]-times[:,None]).ravel();rel=rel[(rel>=-PRE)&(rel<POST)]
                acc+=np.histogram(rel,EDGES)[0]/(len(times)*BW)
        return CEN*1000,std_acc/max(nresp,1),o90_acc/max(nresp,1),nresp,len(vis)
    finally: fh.close()

def img_resp_trace(ds,aid,slap=False):
    """Population-mean dF/F over RESPONSIVE ROIs (reliable positive evoked response)."""
    fh=h5py.File(remfile.File(s3_url(ds,aid)),"r")
    try:
        if slap:
            g=fh["intervals"]["gratings"];O=g["orientation"][:].astype(float);C=g["contrast"][:].astype(float)
            ts=g["start_time"][:];dur=g["stop_time"][:]-g["start_time"][:]
            t_std=ts[np.isclose(O,0.0,atol=0.05)&(C>0)&(dur>=0.3)];t_o90=ts[np.isclose(O,1.571,atol=0.05)&(C>0)&(dur>=0.3)]
            fl=fh["processing"]["ophys"]
            unit_sets=[]
            for dmd,off in [("Fluorescence_DMD1",0.115),("Fluorescence_DMD2",-0.165)]:  # simultaneous DMDs, slight delay
                sub=fl[dmd];key=[k for k in sub if k.endswith("dFF")][0]
                unit_sets.append((sub[key]["data"][:],sub[key]["timestamps"][:]+off))
            RESP=(0.0,0.5)
        else:
            I=fh["intervals"];blk=[k for k in I if "Standard mismatch" in k][0];g=I[blk]
            O=g["Orientation"][:].astype(float);TT=col(g,"TrialType");ts=g["start_time"][:]
            t_std=ts[TT=="standard"];t_o90=ts[TT=="orientation_90"]
            unit_sets=[]
            for pl in [k for k in fh["processing"] if k.startswith("VIS")]:
                grp=fh["processing"][pl];dff=grp["dff_timeseries"]["dff_timeseries"]
                data=dff["data"][:];dts=dff["timestamps"][:]
                try: soma=grp["image_segmentation"]["roi_table"]["is_soma"][:].astype(bool)
                except: soma=np.ones(data.shape[1],bool)
                unit_sets.append((data[:,soma],dts))
            RESP=(0.1,0.8)
        BASE=(-0.3,-0.02)
        def roi_ev(tr,dts,times):
            out=[]
            for t0 in times:
                rb=(dts>=t0+RESP[0])&(dts<t0+RESP[1]);bb=(dts>=t0+BASE[0])&(dts<t0+BASE[1])
                out.append(np.nanmean(tr[rb])-np.nanmean(tr[bb]) if rb.sum() and bb.sum() else np.nan)
            return np.array(out)
        std_acc=np.zeros(len(GRID));o90_acc=np.zeros(len(GRID));nresp=0;ntot=0
        for data,dts in unit_sets:
            for r in range(data.shape[1]):
                ntot+=1; tr=data[:,r]; ev=roi_ev(tr,dts,t_std); ev=ev[np.isfinite(ev)]
                if len(ev)<5 or ss.wilcoxon(ev).pvalue>=0.05 or np.nanmean(ev)<=0: continue
                nresp+=1
                for times,acc in [(t_std,std_acc),(t_o90,o90_acc)]:
                    for t0 in times:
                        i0=np.searchsorted(dts,t0-PRE);i1=np.searchsorted(dts,t0+POST);tt=dts[i0:i1]-t0
                        if len(tt)<3: continue
                        vi=np.interp(GRID,tt,tr[i0:i1],left=np.nan,right=np.nan);m=np.isfinite(vi);acc[m]+=vi[m]
        return GRID*1000,std_acc/(max(nresp,1)*len(t_std)),o90_acc/(max(nresp,1)*len(t_o90)),nresp,ntot
    finally: fh.close()
print("extractors ready")

In [ ]:
#@title Extract responsive-cell traces for every session
import time
traces={m:{} for m in SESSIONS}
for mod,slist in SESSIONS.items():
    for subj,aid in slist:
        t0=time.time()
        try:
            if mod=="ecephys": cen,std,o90,nr,nt=ece_resp_trace(aid)
            else: cen,std,o90,nr,nt=img_resp_trace("001768" if mod=="mesoscope" else "001424",aid,slap=(mod=="slap2"))
            traces[mod][subj]=(cen,std,o90)
            print(f"  {mod}/{subj}: {nr}/{nt} responsive ({time.time()-t0:.0f}s)")
        except Exception as e:
            print(f"  {mod}/{subj}: ERROR {str(e)[:80]}")

## 1 · The same gratings, three modalities, common time axis

Trial- and population-averaged response (responsive cells) to the 0° standard and
90° oddball. The shaded band is the grating (~367 ms). Note how differently the
same event is reported: spikes track onset and offset; jGCaMP8s builds and decays
over ~1–2 s.

In [ ]:
#@title Figure 1 — response PSTHs by modality
import matplotlib.pyplot as plt
MODLAB={"ecephys":"Neuropixels (spikes)","mesoscope":"Mesoscope (jGCaMP8s)","slap2":"SLAP2 (iGluSnFR)"}
YL={"ecephys":"firing rate (Hz)","mesoscope":"mean dF/F","slap2":"mean dF/F"}
CS,CO="#7f7f7f","#c0392b"
fig,axes=plt.subplots(1,3,figsize=(14,4))
for ax,mod in zip(axes,["ecephys","mesoscope","slap2"]):
    subj=list(traces[mod])[0]; cen,std,o90=traces[mod][subj]
    ax.axvspan(0,367,color="#fdf0d5",zorder=0);ax.axvline(0,color="k",lw=0.5,ls=":")
    ax.plot(cen,std,color=CS,lw=1.8,label="0° standard");ax.plot(cen,o90,color=CO,lw=1.8,label="90° oddball")
    ax.set_xlim(-200,2000);ax.set_xlabel("time from onset (ms)");ax.set_ylabel(YL[mod])
    ax.set_title(f"{MODLAB[mod]}\nsub-{subj}",loc="left",fontsize=9)
    if mod=="ecephys": ax.legend(frameon=False,fontsize=8)
fig.tight_layout(); plt.show()

## 2 · Reading the integration window off the data

Left: each modality's evoked response (baseline-subtracted, peak-normalized,
averaged across sessions). Right: the **cumulative fraction** of the evoked
response over time — this is what defines an empirical integration window. The
90%-capture points nearly coincide (~1650–1720 ms), so a single common window of
**~0–1700 ms** captures ≥90% of the response in every modality.

In [ ]:
#@title Figure 2 — normalized overlay + cumulative-response windows
MODCOL={"ecephys":"#2c3e8c","mesoscope":"#c0392b","slap2":"#8e44ad"}
def sess_mean(mod):
    arrs=[]
    for subj,(cen,std,o90) in traces[mod].items():
        v=o90.copy(); v=v-np.nanmean(v[cen<0]); arrs.append(v)
    return list(traces[mod].values())[0][0], np.nanmean(arrs,0)
fig,axes=plt.subplots(1,2,figsize=(13,4.6))
for mod in ["ecephys","mesoscope","slap2"]:
    cen,v=sess_mean(mod);vs=gaussian_filter1d(v,1);pk=np.nanmax(vs[(cen>=0)&(cen<1500)])
    axes[0].plot(cen,vs/pk,color=MODCOL[mod],lw=2)
axes[0].axvspan(0,367,color="#fdf0d5",zorder=0);axes[0].axhline(0,color="0.7",lw=0.6,ls="--")
axes[0].set_xlim(-200,2000);axes[0].set_ylim(-0.6,1.2)
axes[0].set_xlabel("time from onset (ms)");axes[0].set_ylabel("evoked response (peak-norm.)")
axes[0].set_title("Peak-normalized oddball response",loc="left",fontsize=9)
for mod in ["ecephys","mesoscope","slap2"]:
    cen,v=sess_mean(mod);vs=gaussian_filter1d(np.clip(v,0,None),1)
    post=cen>=0;c=np.cumsum(vs[post]);c=c/c[-1];tt=cen[post]
    n=len(traces[mod]); axes[1].plot(tt,c,color=MODCOL[mod],lw=2,label=f"{MODLAB[mod]} (n={n})")
    t90=tt[np.searchsorted(c,0.9)]; print(f"{mod}: 90% capture at {t90:.0f} ms")
axes[1].axhline(0.9,color="0.85",lw=0.6,ls=":");axes[1].set_xlim(0,2000);axes[1].set_ylim(0,1.05)
axes[1].set_xlabel("time from onset (ms)");axes[1].set_ylabel("cumulative fraction of response")
axes[1].set_title("Cumulative response → integration window",loc="left",fontsize=9)
axes[1].legend(frameon=False,fontsize=7.5,loc="lower right")
fig.tight_layout(); plt.show()

## Takeaway

- The timescale mismatch between spiking and imaging is a **measurement** effect,
  not biology, and it is solved empirically — no deconvolution needed.
- **Restrict to responsive cells.** Population-mean imaging traces (especially
  SLAP2 iGluSnFR) are dominated by ongoing activity and give spurious peak times;
  the responsive-cell subset recovers a real stimulus-locked response.
- Once gated, peak times order by indicator kinetics (spikes ~70 ms, iGluSnFR
  ~200 ms, jGCaMP8s ~880 ms), but the 90%-cumulative-response windows nearly
  coincide (~0–1700 ms).
- **Practical recipe:** to put oddball responses on one footing, integrate the
  evoked response over ~0–1700 ms (or each modality's own 90% window; they nearly
  coincide) on responsive cells. This is orthogonal to — and does not fix — the
  tuning-dominance confound in mesoscope feature-DvI, which is a separate,
  biological issue documented in the README.

---
*Generated for `ai_oscp_neuro`. Data: DANDI 001637 / 001768 / 001424 (OpenScope
Community Predictive Processing, Allen Institute for Neural Dynamics). Streams
anonymously — no credentials needed.*